In [ ]:
# ========== 导入：多 Agent 对话需要的 OpenAI / JSON / 环境变量 / 展示 ==========

# 从 openai 导入 OpenAI：同时用于云端与 Ollama 兼容端点
from openai import OpenAI
# 导入 json：把对话历史序列化进 user prompt
import json 
# 导入 os：读环境变量里的 API Key
import os 
# 从 dotenv 导入 load_dotenv：加载 .env，避免把密钥写进笔记本
from dotenv import load_dotenv
# 从 IPython.display 导入 Markdown：用标题样式展示每位 Agent 的发言
from IPython.display import Markdown


In [ ]:
# ========== 加载密钥：从 .env 读入 OPENAI_API_KEY ==========

# override=True：.env 中的值覆盖进程里已有同名环境变量
load_dotenv(override=True)
# 取出云端 OpenAI 的 API Key（后面创建 openAI 客户端会用到环境变量）
openai_api_key = os.getenv("OPENAI_API_KEY")


In [ ]:
# ========== 密钥体检：有 Key 就打印前缀，没有就提示未设置 ==========

# 若已读到密钥：只展示前 8 个字符，避免把完整 Key 打到输出里
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    # 排错文案保持英文原样
    print("OpenAI API Key not set")


In [ ]:
# ========== 云端客户端：默认 OpenAI()（读环境变量中的 Key） ==========

# 变量名是 openAI（驼峰），后面 agent_info 里会引用它
openAI = OpenAI()


In [ ]:
# ========== 准备本地模型：拉取 llama3.2（需本机已安装并启动 Ollama） ==========

# 笔记本 shell 魔法：在终端执行 ollama pull；模型名字符串保持原样
!ollama pull llama3.2


In [ ]:
# ========== 本地 Ollama 客户端：OpenAI 兼容 /v1 端点 ==========

# Ollama 兼容接口地址（默认端口 11434）；URL 保持原样
ollama_url="http://localhost:11434/v1"
# api_key 对本地 Ollama 通常任意；base_url 指向上面的兼容地址
ollama = OpenAI(api_key="ollama", base_url=ollama_url)


In [ ]:
# ========== 三角色各自的模型 id：云端 / 本地混用 ==========

# Alex：云端较新的小模型（model id 字符串禁止改写）
alex_model = "gpt-5.4-mini-2026-03-17"
# Blake：走本地 Ollama 的 llama3.2
blake_model = "llama3.2"
# Charlie：云端 gpt-4o-mini
charlie_model = "gpt-4o-mini"


In [ ]:
# ========== 三角色 System Prompt：度假偏好人设（影响回复风格，正文不翻译） ==========

# Alex：奢华、舒适、偏精英语气
alex_prompt = """
Philosophy: Vacation = Pampering, Status, and Comfort.
Top Pick: The Maldives or Monaco.
Traits: Prefers 5-star resorts, private villas, and "Instagrammable" aesthetics. Values convenience and exclusivity above all.
Tone: Sophisticated, slightly elitist, dismisses anything "budget" or "exhausting."
"""

# Blake：运动、挑战、探索；嫌弃「躺平式」度假
blake_prompt = """
Philosophy: Vacation = Movement, Challenge, and Exploration.
Top Pick: South Africa (Safaris) or Dahab (Diving/Hiking).
Traits: Hates staying in one place. Wants to hike, dive, or skydive. Prefers hostels or camps over fancy hotels.
Tone: Energetic, blunt, finds Alex’s style "boring" and "lazy."
"""

# Charlie：历史、氛围、慢旅行；常在争论里扮演调解但坚持文化向
charlie_prompt = """
Philosophy: Vacation = History, Soul, and Quiet Beauty.
Top Pick: Santorini (Greece) or Tuscany (Italy).
Traits: Loves walking through old streets, visiting museums, and local cafes. Values "The Vibe" and authenticity.
Tone: Calm, poetic, intellectual. Acts as the mediator but remains stubborn about their love for "culture" over "resorts" or "extreme sports."
"""


In [ ]:
# ========== 工具函数：把对话历史转成带缩进的 JSON 字符串 ==========

def convert_to_json (chat) : 
  # ensure_ascii=False：保留非 ASCII 字符；indent=2：便于塞进 prompt 阅读
  return json.dumps (chat, ensure_ascii= False , indent= 2)


In [ ]:
# ========== 拼 User Prompt：告诉模型「你是谁 + 目前聊到哪 + 请接着说」 ==========

def build_user_prompt (agent_name , history_chat ) : 
  # 先把历史列表序列化，方便模型看清 speaker/content
  chat_to_json = convert_to_json (history_chat)
  # f-string 里的英文指令保持原样（限制约 150 词，更像真人短聊）
  user_prompt = f"""
  you are {agent_name} in conversation with the two other participants .
  this conversation is follow as : 
  {chat_to_json}

  now respond as {agent_name} 
  Try to reply with a maximum of 150 words to sound more human.
  """
  return user_prompt


In [ ]:
# ========== 调用单个 Agent：system 定人设，user 带历史，返回一句回复 ==========


def call_agent (model , system_prompt , user_prompt , client) : 
  # 使用传入的 client（可能是 openAI 或 ollama）发非流式 chat.completions
  responses = client.chat.completions.create ( 
    model = model   ,
    messages = [ 
      {"role" : "system" , "content" : system_prompt} , 
      {"role" : "user" , "content" : user_prompt}
    ],
  )

  # 取出第一条 choice 的文本内容
  return responses.choices[0].message.content


In [ ]:
# ========== 多人轮转对话：共享历史，按 agent_info 顺序每人发言 ==========



# 对话历史：先塞一句 Alex 的开场（目的地问题）；结构 {speaker, content}



agent_chat = [ 
  {"speaker" : "Alex" , "content" : "What is your destination for vacation?"}
]
# 三位参与者的配置：名字、模型、system prompt、对应客户端
agent_info = [ 
  {"name" :"blake" , "model_name" :blake_model , "system_prompt" : blake_prompt , "client" : ollama },
  {"name" :"charlie" , "model_name" :charlie_model , "system_prompt" : charlie_prompt , "client" : openAI},
  {"name" :"Alex" , "model_name" :alex_model , "system_prompt" : alex_prompt , "client" :openAI }
]

# 外层 4 轮；内层按 agent_info 顺序轮流发言（共 12 次模型调用）
for _ in range (4) : 
  for agent in agent_info : 
    # 根据当前历史，为该 agent 生成 user prompt
    User = build_user_prompt(agent["name"] ,agent_chat)
    # 调用对应 backend / model，拿到回复字符串
    response = call_agent (agent["model_name"],agent["system_prompt"] , User ,agent["client"])
    # 把新发言追加进共享历史，供下一位阅读
    agent_chat.append ({"speaker" : agent["name"] , "content" : response })
    # 立刻用 Markdown 标题展示这位的发言
    display(Markdown (f"### {agent["name"]} :\n{response} "))

In [ ]:
# ========== 收尾：展示完整对话列表（字典数组） ==========

# 在笔记本里打印/展示 agent_chat，便于回顾全文
display (agent_chat)
